## Introduction

Although adaptation of methods used in physics to finance is common, known as the field of econophysics, computational mechanics influence remains limited in financial applications. The only attempt we are aware is of [Park et al (2007)](https://www.sciencedirect.com/science/article/abs/pii/S0378437107000271). They estimated discrete $\epsilon$-machines on a one-year rolling windows of intraday S\&P500 prices, observed that the number of discovered causal states decreased between 1983 and 2006, and concluded that over time the new information is reflected in market prices quicker. However, their methodology has a substantial shortcoming - discretizing market prices into binary up/down movements leads to a loss of information. By applying the framework of kernel $\epsilon$-machines of [Brodu and Crutchfield (2022)](https://doi.org/10.1063/5.0062829), we are able to work directly with original continuous data.

We came up with two ideas how computational mechanics can contribute to the financial literature:
1. Statistical complexity $C_\mu$ can be employed to measure market efficiency
2. Causal state surprisal can be employed as a early warning signal for mean reversion, breakout or stress in the financial time-series

In this notebook we consider the first application of the above mentioned.

## Efficient and Adaptive Markets

Considerations on market efficiency date back to Louis Bachelier. In his [PhD disertation](https://www.investmenttheory.org/uploads/3/4/8/2/34825752/emhbachelier.pdf), he claimed that *past, present and even discounted future events are reflected in market price*.

In [1970 Eugene Fama](https://doi.org/10.2307/2325486) synthetized literature and empirical evidence, and introduced a taxonomy of market efficiency. Namely, he distinguished weak, semi-strong and strong types of market efficiency. Weak-form assumes that past prices are reflected in future prices, semi-strong form assumes that all publicly available data is reflected in an asset price, whereas strong-form assumes, all data including insider information is already reflected in prices.

[Andrew Lo, in 2004](https://ssrn.com/abstract=602222), reconciled efficient markets theory with behavioral finance by introducing **adaptive market hypothesis**. He argues that market efficiency is context dependent, time varying and shaped by local market peculiarities rather than by a fixed stationary benchmark.

By applying computational mechanics to financial time-series we would like to quantify the adaptive market hypethesis by investigating whether:
- (Q1) Developed markets have lower statistical complexity $C_\mu$ than emerging markets.
- (Q2) Industries or sectors with lower liquidity, greater segmentation, or higher retail participation have higher statistical complexity $C_\mu$.
- (Q3) Statistical complexity $C_\mu$ declines as the sampling interval is coarsened.
- (Q4) Statistical complexity $C_\mu$ varies over time, and rises during market stress.

## Data Preparation

We download financial time-series from Yahoo Finance through Python's API `yfinance`.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

As a starting point, to ensure data homogeinity, we use liquid US-listed ETFs. However, the future research could use local market data.

As a proxy for developed markets we choose the following ETFs:
 - `SPY` - United States
 - `EWU` - United Kingdom
 - `EWG` - Germany
 - `EWJ` - Japan

As a proxy for emerging markets we choose the following ETFs:
 - `MCHI` - China
 - `INDA` - India
 - `EWZ` - Brazil
 - `EPOL` - Poland

We begin with 1d interval on a balanced panel.

In [2]:
country_groups = {
    "SPY": "developed",
    "EWU": "developed",
    "EWG": "developed",
    "EWJ": "developed",
    "MCHI": "emerging",
    "INDA": "emerging",
    "EWZ": "emerging",
    "EPOL": "emerging",
}

country_tickers = list(country_groups.keys())

country_panel = yf.download(
    tickers=country_tickers,
    start="2000-01-01",
    end="2025-12-31",
    interval="1d",
    progress=False,
 )

country_panel = country_panel["Close"].copy()
country_panel_balanced = country_panel.dropna(how="any")

display(country_panel_balanced.head())

Ticker,EPOL,EWG,EWJ,EWU,EWZ,INDA,MCHI,SPY
Date,,,,,,,,
2012-02-03,17.675797,16.182777,29.863401,19.932020,36.147049,23.198490,34.656551,104.795776
2012-02-06,17.777117,16.117844,29.770761,19.852106,36.072922,23.016169,34.173851,104.725693
2012-02-07,17.918949,16.240501,30.079594,19.932020,36.475327,22.807798,34.166306,104.990524
2012-02-08,17.898685,16.327068,30.203115,19.897774,36.496513,23.059578,34.928062,105.302086
2012-02-09,17.736586,16.406435,30.110479,19.932020,36.438263,22.998802,34.897907,105.434494
